# Twitter Sentiment Analysis Using LSTM

# Import Libraries

In [4]:

import numpy as np
import pandas as pd
import re

from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

# Load Dataset

In [5]:
df = pd.read_csv(
    "/content/twitter_training.csv",
    header=None
)

In [6]:
print(df.head())

      0            1         2  \
0  2401  Borderlands  Positive   
1  2401  Borderlands  Positive   
2  2401  Borderlands  Positive   
3  2401  Borderlands  Positive   
4  2401  Borderlands  Positive   

                                                   3  
0  im getting on borderlands and i will murder yo...  
1  I am coming to the borders and I will kill you...  
2  im getting on borderlands and i will kill you ...  
3  im coming on borderlands and i will murder you...  
4  im getting on borderlands 2 and i will murder ...  


# Add Column Names

In [7]:

df.columns = ['id', 'game', 'sentiment', 'text']

# Check columns
print(df.columns)


Index(['id', 'game', 'sentiment', 'text'], dtype='object')


# Data Preprocessing

## Keep Only Positve And Negative Tweets

In [8]:
df = df[
    (df['sentiment'] == 'Positive') |
    (df['sentiment'] == 'Negative')
]

## Select  Texts And Labels

In [9]:

texts = df['text']
labels = df['sentiment']

## Convert Labels Into Numbers

In [10]:
labels = labels.map({
    'Positive': 1,
    'Negative': 0
})

# Convert datatype into integer
labels = labels.astype(int)

## Text Cleaning

In [11]:

def clean_text(text):

    # Convert into lowercase
    text = str(text).lower() # Convert to string to handle non-string types safely

    # Remove links
    text = re.sub(r'http\S+', '', text)

    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    return text


# Apply cleaning
texts = texts.apply(clean_text)


# Tokenization

In [12]:

vocab_size = 10000

tokenizer = Tokenizer(num_words=vocab_size)

tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)

# Padding Sequences

In [13]:
maxlen = 200

X = pad_sequences(sequences, maxlen=maxlen)

y = labels


# Train Test Split

In [14]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# Build LSTM Model

In [15]:
model = Sequential()

## Embedding Layer

In [16]:
model.add(Embedding(vocab_size, 64))


## LSTM Layer

In [17]:
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))

## Dense Output Layer

In [18]:

model.add(Dense(1, activation='sigmoid'))

# Compile Model

In [19]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train Model

In [20]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_data=(X_test, y_test)
)

Epoch 1/5
436/436 ━━━━━━━━━━━━━━━━━━━━ 156s 349ms/step - accuracy: 0.7934 - loss: 0.4318 - val_accuracy: 0.8861 - val_loss: 0.2686
Epoch 2/5
436/436 ━━━━━━━━━━━━━━━━━━━━ 203s 351ms/step - accuracy: 0.9108 - loss: 0.2147 - val_accuracy: 0.9196 - val_loss: 0.1930
Epoch 3/5
436/436 ━━━━━━━━━━━━━━━━━━━━ 155s 355ms/step - accuracy: 0.9361 - loss: 0.1490 - val_accuracy: 0.9200 - val_loss: 0.1931
Epoch 4/5
436/436 ━━━━━━━━━━━━━━━━━━━━ 202s 356ms/step - accuracy: 0.9507 - loss: 0.1138 - val_accuracy: 0.9287 - val_loss: 0.1776
Epoch 5/5
436/436 ━━━━━━━━━━━━━━━━━━━━ 151s 346ms/step - accuracy: 0.9540 - loss: 0.0999 - val_accuracy: 0.9266 - val_loss: 0.1937


# Evaluate Model

In [21]:

loss, accuracy = model.evaluate(X_test, y_test)

print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")


218/218 ━━━━━━━━━━━━━━━━━━━━ 10s 46ms/step - accuracy: 0.9266 - loss: 0.1937
Test Loss: 0.1937
Test Accuracy: 0.9266


#  Make Prediction

In [22]:
sample_text = ["this product is amazing"]

# Convert text to sequence
sample_seq = tokenizer.texts_to_sequences(sample_text)

# Padding
sample_pad = pad_sequences(sample_seq, maxlen=maxlen)

# Prediction
prediction = model.predict(sample_pad)

print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 305ms/step
[[0.99557674]]


# Final Result

In [23]:

if prediction[0][0] > 0.5:
    print("Positive Tweet 😊")
else:
    print("Negative Tweet 😔")

Positive Tweet 😊
